# V11 Mike

## Baseline: V11 Lima (best unseen F1 = 0.3890 on PC1, best unseen acc = 0.5971 on PC2)

Mike targets the four diagnosed weak spots of Lima with focused architectural improvements.
Every other hyperparameter is kept identical to Lima.

| Component | Lima | Mike | Reason |
|---|---|---|---|
| **fuse_dim** | 64 | **128** | More capacity for 6 acoustically similar classes. At 64 dims the boundary between Healthy/COPD and Bronchiolitis/URTI is too coarse |
| **TemporalAttention** | Single head `Linear(hidden,1)` | **4-head Multi-Head** | Single scalar per time step misses concurrent temporal patterns. 4 heads let the model track wheeze, crackle, rhonchi patterns simultaneously |
| **Focal Loss gamma** | Uniform 1.5 all classes | **Per-class: Bronchiectasis=2.0, Bronchiolitis=2.0, COPD=1.0, Healthy=1.5, Pneumonia=1.5, URTI=2.0** | Hard minority classes need stronger penalty. COPD already learned well and needs less focusing |
| **Classifier head** | 64->128->6 (trumpet) | **128->64->6 (funnel)** | Expansion before collapse wastes capacity. Progressive compression is the correct design |
| GatedFusion gate | Constrained [0.35, 0.65] | Same | Both machines converged to same pattern in Lima |
| LayerNorm after LSTM | Present | Same | Working well |
| Chroma inter-dropout | Fixed | Same | Fixed in Lima, symmetric now |
| LSTM depth | 2-layer + inter-dropout | Same | Proven since Charlie |
| LR Scheduler | ReduceLROnPlateau | Same | Proven better than CosineAnnealing |
| Class weights | compute_class_weight balanced | Same | Still needed alongside per-class gamma |
| AdamW lr | 1e-4 | Same | No evidence to change |
| weight_decay | 1e-4 | Same | No evidence to change |
| batch_size | 64 | Same | Stable gradients |
| patience | 25 | Same | Solid across all versions |
| TARGET_COUNT | 500 | Same | Consistent cap per class |

**Mike = Lima + fuse_dim 128 + 4-head TemporalAttention + per-class focal gamma + funnel classifier head**

## 1. Imports and Reproducibility

In [ ]:
import os
import random
from dataclasses import dataclass

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2. Paths and Feature Parameters

In [ ]:
import numpy as np
import pandas as pd
import librosa as lb

BASE_DIR  = "../../dataset"
TRAIN_DIR = os.path.join(BASE_DIR, "train-segments")
TEST_DIR  = os.path.join(BASE_DIR, "test-segments")
VAL_DIR   = os.path.join(BASE_DIR, "val-segments")
TRAIN_CSV = os.path.join(BASE_DIR, "train_segments.csv")
TEST_CSV  = os.path.join(BASE_DIR, "test_segments.csv")
VAL_CSV   = os.path.join(BASE_DIR, "val_segments.csv")

SR              = 4000
N_MFCC          = 40
TARGET_MFCC_T   = 679
TARGET_CHROMA_T = 259

## 3. Feature Extraction Helpers

In [ ]:
def extract_mfcc_and_chroma(y, sr):
    mfcc   = lb.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    chroma = lb.feature.chroma_stft(y=y, sr=sr)
    return mfcc, chroma


def pad_or_crop(feat, target_T):
    F, T = feat.shape
    if T > target_T:
        return feat[:, :target_T]
    if T < target_T:
        return np.pad(feat, ((0, 0), (0, target_T - T)), mode="constant")
    return feat

## 4. Build Features from CSV

In [ ]:
def build_features_from_csv(csv_path, audio_dir):
    df          = pd.read_csv(csv_path)
    mfcc_list   = []
    chroma_list = []
    labels      = []
    diseases    = sorted(df["disease"].unique())
    label_map   = {d: i for i, d in enumerate(diseases)}

    for _, row in df.iterrows():
        filename  = os.path.basename(row["segment_file"])
        label     = row["disease"]
        file_path = os.path.join(audio_dir, filename)
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"Missing file: {file_path}")
        y, sr        = lb.load(file_path, sr=SR)
        mfcc, chroma = extract_mfcc_and_chroma(y, sr)
        mfcc         = pad_or_crop(mfcc,   TARGET_MFCC_T)
        chroma       = pad_or_crop(chroma, TARGET_CHROMA_T)
        mfcc_list.append(mfcc)
        chroma_list.append(chroma)
        labels.append(label_map[label])

    return np.stack(mfcc_list), np.stack(chroma_list), np.array(labels), label_map

## 5. Load Data

In [ ]:
X_mfcc_train, X_chroma_train, y_train, label_map = build_features_from_csv(TRAIN_CSV, TRAIN_DIR)
X_mfcc_test,  X_chroma_test,  y_test,  _         = build_features_from_csv(TEST_CSV,  TEST_DIR)
X_mfcc_val,   X_chroma_val,   y_val,   _         = build_features_from_csv(VAL_CSV,   VAL_DIR)

## 6. Convert to Tensors

In [ ]:
X_mfcc_train   = torch.from_numpy(X_mfcc_train).float()
X_chroma_train = torch.from_numpy(X_chroma_train).float()
y_train        = torch.from_numpy(y_train).long()

X_mfcc_test    = torch.from_numpy(X_mfcc_test).float()
X_chroma_test  = torch.from_numpy(X_chroma_test).float()
y_test         = torch.from_numpy(y_test).long()

X_mfcc_unseen   = torch.from_numpy(X_mfcc_val).float()
X_chroma_unseen = torch.from_numpy(X_chroma_val).float()
y_unseen        = torch.from_numpy(y_val).long()

num_classes = len(label_map)
print("Train:",  X_mfcc_train.shape,  X_chroma_train.shape,  y_train.shape)
print("Test: ",  X_mfcc_test.shape,   X_chroma_test.shape,   y_test.shape)
print("Unseen:", X_mfcc_unseen.shape, X_chroma_unseen.shape, y_unseen.shape)
print("Label map:", label_map)

## 7. Dataset and DataLoader

In [ ]:
class DualFeatureDataset(Dataset):
    def __init__(self, X_mfcc, X_chroma, y):
        assert len(X_mfcc) == len(X_chroma) == len(y)
        self.X_mfcc   = X_mfcc
        self.X_chroma = X_chroma
        self.y        = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_mfcc[idx], self.X_chroma[idx], self.y[idx]


unseen_loader = DataLoader(
    DualFeatureDataset(X_mfcc_unseen, X_chroma_unseen, y_unseen),
    batch_size=32, shuffle=False, num_workers=0, pin_memory=True
)

## 8. Model Architecture

### Changes from Lima

**1. fuse_dim increased 64 -> 128.**
Lima compressed every feature vector to 64 dimensions. With 6 acoustically similar
classes this is too tight. The boundary between Healthy/COPD and Bronchiolitis/URTI
cannot be cleanly represented at 64 dims. Doubling to 128 gives each fusion stage
genuine representational capacity.

**2. Single-head TemporalAttention replaced with 4-head MultiHeadTemporalAttention.**
Lima used Linear(hidden_dim, 1) — one scalar per time step, applied uniformly
across all feature dimensions. This misses concurrent temporal patterns:
wheeze is sustained, crackles are transient, rhonchi are rhythmic.
4 independent heads each learn a different temporal focus pattern and their
outputs are concatenated then projected back to fuse_dim.

**3. Classifier head changed from trumpet (64->128->6) to funnel (128->64->6).**
Lima's 64->128 expansion bloated a compressed vector with no new information.
Mike's 128->64 compression progressively distills the richer fuse_dim=128
representation down to the 6-class decision — the correct funnel design.

**4. GatedFusion gate constraint [0.35, 0.65] unchanged from Lima.**
Both machines converged to the same stable pattern in Lima for the first time.
This is proven and retained.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GatedFusion(nn.Module):
    """
    Constrained gated fusion — weight w clamped to [0.35, 0.65].
    Unchanged from Lima. Both PC1 and PC2 converged to the same stable
    gate pattern for the first time in Lima, confirming the constraint
    produces a reproducible learned fusion equilibrium.
    """
    def __init__(self, dim: int, hidden: int = 128, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim * 2, hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden, dim),
        )

    def forward(self, a: torch.Tensor, b: torch.Tensor):
        w     = torch.clamp(
            torch.sigmoid(self.net(torch.cat([a, b], dim=1))),
            min=0.35, max=0.65
        )
        fused = w * a + (1.0 - w) * b
        return fused, w


class MultiHeadTemporalAttention(nn.Module):
    """
    4-head temporal attention over LSTM time steps. [NEW in Mike]

    Lima's single-head TemporalAttention used one scalar weight per time
    step (Linear(hidden_dim, 1)), treating all feature dimensions equally
    at each frame. This misses concurrent temporal patterns in respiratory
    audio — wheeze is sustained, crackles are transient, rhonchi rhythmic.

    4 independent attention heads each learn a different temporal focus
    pattern. Their outputs are concatenated [B, hidden_dim] and projected
    back via a linear layer, giving multi-pattern temporal awareness
    without increasing output dimensionality.

    Requires: hidden_dim % num_heads == 0
    head_dim = hidden_dim // num_heads
    """
    def __init__(self, hidden_dim: int, num_heads: int = 4):
        super().__init__()
        assert hidden_dim % num_heads == 0, \
            f"hidden_dim ({hidden_dim}) must be divisible by num_heads ({num_heads})"
        self.num_heads = num_heads
        self.head_dim  = hidden_dim // num_heads
        self.head_attns = nn.ModuleList([
            nn.Linear(self.head_dim, 1) for _ in range(num_heads)
        ])
        self.proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, hidden_dim]
        chunks    = x.split(self.head_dim, dim=2)  # list of [B, T, head_dim]
        head_outs = []
        for i, chunk in enumerate(chunks):
            scores  = self.head_attns[i](chunk).squeeze(-1)   # [B, T]
            weights = torch.softmax(scores, dim=1)            # [B, T]
            out     = (weights.unsqueeze(-1) * chunk).sum(1)  # [B, head_dim]
            head_outs.append(out)
        concat = torch.cat(head_outs, dim=1)   # [B, hidden_dim]
        return self.proj(concat)               # [B, hidden_dim]


class DualChannelCNNLSTM_WithAttentionFusion(nn.Module):

    def __init__(self, num_classes: int = 6, dropout_p: float = 0.4, fuse_dim: int = 128):
        super().__init__()
        self.fuse_dim = fuse_dim

        # MFCC CNN — unchanged from Lima
        self.mfcc_cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.BatchNorm2d(32), nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.BatchNorm2d(64), nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.BatchNorm2d(128), nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
        )

        # Chroma CNN — unchanged from Lima
        self.chroma_cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.BatchNorm2d(32), nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.BatchNorm2d(64), nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(inplace=True),
            nn.BatchNorm2d(128), nn.MaxPool2d(2, 2), nn.Dropout2d(0.2),
        )

        self.gap2d           = nn.AdaptiveAvgPool2d((1, 1))
        # CNN projections output fuse_dim=128 [CHANGED from Lima's 64]
        self.mfcc_cnn_proj   = nn.Linear(128, fuse_dim)
        self.chroma_cnn_proj = nn.Linear(128, fuse_dim)

        # MFCC LSTM — 2 layers + dropout + MultiHeadTemporalAttention + LayerNorm
        # LSTM2 output = fuse_dim=128 [CHANGED from Lima's 64]
        # MultiHeadTemporalAttention replaces single-head [CHANGED]
        self.mfcc_lstm1     = nn.LSTM(40,  128,      batch_first=True)
        self.mfcc_lstm_drop = nn.Dropout(p=0.3)
        self.mfcc_lstm2     = nn.LSTM(128, fuse_dim, batch_first=True)
        self.mfcc_attn      = MultiHeadTemporalAttention(fuse_dim, num_heads=4)  # CHANGED
        self.mfcc_lstm_norm = nn.LayerNorm(fuse_dim)

        # Chroma LSTM — identical structure to MFCC LSTM
        self.chroma_lstm1     = nn.LSTM(12,  128,      batch_first=True)
        self.chroma_lstm_drop = nn.Dropout(p=0.3)
        self.chroma_lstm2     = nn.LSTM(128, fuse_dim, batch_first=True)
        self.chroma_attn      = MultiHeadTemporalAttention(fuse_dim, num_heads=4)  # CHANGED
        self.chroma_lstm_norm = nn.LayerNorm(fuse_dim)

        # Fusion modules — constrained GatedFusion, now at fuse_dim=128
        self.fuse_cnn_modality  = GatedFusion(dim=fuse_dim, hidden=128, dropout=0.1)
        self.fuse_lstm_modality = GatedFusion(dim=fuse_dim, hidden=128, dropout=0.1)
        self.fuse_extractor     = GatedFusion(dim=fuse_dim, hidden=128, dropout=0.1)

        # Classifier head — FUNNEL 128->64->6 [CHANGED from Lima's 64->128->6]
        # Lima:  64 -> 128 -> 6  (trumpet — expands then collapses, wastes capacity)
        # Mike: 128 ->  64 -> 6  (funnel  — progressively compresses, correct design)
        self.fc1     = nn.Linear(fuse_dim, 64)    # 128 -> 64
        self.dropout = nn.Dropout(p=dropout_p)
        self.fc_out  = nn.Linear(64, num_classes) # 64  -> 6

    def forward(self, mfcc: torch.Tensor, chroma: torch.Tensor, return_attn: bool = False):

        # CNN pathway
        mfcc_cnn_vec   = self.gap2d(self.mfcc_cnn(mfcc.unsqueeze(1))).flatten(1)
        mfcc_cnn_vec   = self.mfcc_cnn_proj(mfcc_cnn_vec)
        chroma_cnn_vec = self.gap2d(self.chroma_cnn(chroma.unsqueeze(1))).flatten(1)
        chroma_cnn_vec = self.chroma_cnn_proj(chroma_cnn_vec)

        # MFCC LSTM pathway
        x, _ = self.mfcc_lstm1(mfcc.transpose(1, 2))
        x     = self.mfcc_lstm_drop(x)
        x, _ = self.mfcc_lstm2(x)
        mfcc_lstm_vec = self.mfcc_lstm_norm(self.mfcc_attn(x))

        # Chroma LSTM pathway
        y, _ = self.chroma_lstm1(chroma.transpose(1, 2))
        y     = self.chroma_lstm_drop(y)
        y, _ = self.chroma_lstm2(y)
        chroma_lstm_vec = self.chroma_lstm_norm(self.chroma_attn(y))

        # Intra-modal fusion
        cnn_fused,  w_cnn  = self.fuse_cnn_modality(mfcc_cnn_vec,   chroma_cnn_vec)
        lstm_fused, w_lstm = self.fuse_lstm_modality(mfcc_lstm_vec, chroma_lstm_vec)

        # Inter-modal fusion
        final_fused, w_inter = self.fuse_extractor(cnn_fused, lstm_fused)

        # Funnel classifier: 128 -> 64 -> 6
        z      = F.relu(self.fc1(final_fused))
        z      = self.dropout(z)
        logits = self.fc_out(z)

        if return_attn:
            return logits, {
                "w_cnn_mean":   float(w_cnn.mean().item()),
                "w_lstm_mean":  float(w_lstm.mean().item()),
                "w_inter_mean": float(w_inter.mean().item()),
            }
        return logits

## 9. Early Stopping

In [ ]:
@dataclass
class EarlyStopping:
    patience:       int   = 7
    min_delta:      float = 0.0
    best:           float = float("inf")
    num_bad_epochs: int   = 0
    stopped:        bool  = False

    def step(self, current: float) -> bool:
        improved = (self.best - current) > self.min_delta
        if improved:
            self.best           = current
            self.num_bad_epochs = 0
        else:
            self.num_bad_epochs += 1
            if self.num_bad_epochs >= self.patience:
                self.stopped = True
        return self.stopped


checkpoint_path = "best_model.pt"

## 10. Training Loop (run_epoch)

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, balanced_accuracy_score


def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    all_losses, all_preds, all_targets = [], [], []
    attn_w_cnn, attn_w_lstm, attn_w_inter = [], [], []

    for mfcc, chroma, y in loader:
        mfcc   = mfcc.to(device,   non_blocking=True)
        chroma = chroma.to(device, non_blocking=True)
        y      = y.to(device,      non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits, attn = model(mfcc, chroma, return_attn=True)
            loss = criterion(logits, y)

        if is_train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        all_losses.append(loss.item())
        preds = torch.argmax(logits, dim=1)
        all_preds.append(preds.detach().cpu().numpy())
        all_targets.append(y.detach().cpu().numpy())
        attn_w_cnn.append(attn["w_cnn_mean"])
        attn_w_lstm.append(attn["w_lstm_mean"])
        attn_w_inter.append(attn["w_inter_mean"])

    all_preds   = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    acc     = accuracy_score(all_targets, all_preds)
    bal_acc = balanced_accuracy_score(all_targets, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, average="macro", zero_division=0
    )
    return (
        float(np.mean(all_losses)),
        acc, bal_acc, prec, rec, f1,
        float(np.mean(attn_w_cnn)),
        float(np.mean(attn_w_lstm)),
        float(np.mean(attn_w_inter)),
    )

## 11. 10-Fold Cross-Validation

### Training configuration

All hyperparameters identical to Lima except per-class focal gamma.

**Per-class gamma assignment (based on Lima unseen F1 per class):**
- Class 0 Bronchiectasis: gamma=2.0 (F1=0.20 in Lima — hardest)
- Class 1 Bronchiolitis:  gamma=2.0 (F1=0.26 in Lima — stuck)
- Class 2 COPD:           gamma=1.0 (F1=0.71 in Lima — already learned well)
- Class 3 Healthy:        gamma=1.5 (F1=0.25 in Lima — moderate)
- Class 4 Pneumonia:      gamma=1.5 (F1=0.50 in Lima — moderate)
- Class 5 URTI:           gamma=2.0 (F1=0.19-0.41 in Lima — inconsistent)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix
import seaborn as sns

epochs       = 500
n_splits     = 10
batch_size   = 64
TARGET_COUNT = 500

# -------------------------------------------------------------------
# Per-class Focal Loss  [CHANGED from Lima's uniform gamma=1.5]
#
# Class order (alphabetical, matches label_map):
#   0=Bronchiectasis, 1=Bronchiolitis, 2=COPD, 3=Healthy, 4=Pneumonia, 5=URTI
# -------------------------------------------------------------------
CLASS_GAMMAS = torch.tensor([2.0, 2.0, 1.0, 1.5, 1.5, 2.0], dtype=torch.float32)


class PerClassFocalLoss(nn.Module):
    """
    Focal Loss with per-class gamma values.
    Each sample's focal weight uses the gamma of its true class:
        focal_weight = (1 - p_t) ^ gamma_c
    This lets hard minority classes receive stronger gradient focus
    than easy majority classes, independently of class frequency weights.
    """
    def __init__(self, class_gammas: torch.Tensor, weight=None):
        super().__init__()
        self.register_buffer("class_gammas", class_gammas)
        self.weight = weight

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_prob      = F.log_softmax(logits, dim=1)
        prob          = torch.exp(log_prob)
        log_prob_true = log_prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        prob_true     = prob.gather(1, targets.unsqueeze(1)).squeeze(1)
        gamma_per_sample = self.class_gammas[targets]
        focal_weight     = (1.0 - prob_true) ** gamma_per_sample
        loss = -focal_weight * log_prob_true
        if self.weight is not None:
            loss = loss * self.weight[targets]
        return loss.mean()


def cap_per_class(mfcc, chroma, y, cap=TARGET_COUNT, seed=SEED):
    rng      = np.random.default_rng(seed)
    keep_idx = []
    for cls in np.unique(y):
        cls_idx = np.where(y == cls)[0]
        if len(cls_idx) > cap:
            cls_idx = rng.choice(cls_idx, cap, replace=False)
        keep_idx.append(cls_idx)
    keep_idx = np.sort(np.concatenate(keep_idx))
    return mfcc[keep_idx], chroma[keep_idx], y[keep_idx]


X_mfcc_train_np   = X_mfcc_train.numpy()   if isinstance(X_mfcc_train,   torch.Tensor) else X_mfcc_train
X_chroma_train_np = X_chroma_train.numpy() if isinstance(X_chroma_train, torch.Tensor) else X_chroma_train
y_train_np        = y_train.numpy()        if isinstance(y_train,        torch.Tensor) else y_train
X_mfcc_test_np    = X_mfcc_test.numpy()    if isinstance(X_mfcc_test,    torch.Tensor) else X_mfcc_test
X_chroma_test_np  = X_chroma_test.numpy()  if isinstance(X_chroma_test,  torch.Tensor) else X_chroma_test
y_test_np         = y_test.numpy()         if isinstance(y_test,         torch.Tensor) else y_test

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

fold_summaries  = []
best_fold_loss  = float("inf")
best_fold_state = None
all_cv_preds    = []
all_cv_targets  = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(y_test_np)), y_test_np), start=1):
    print(f"\n========== Fold {fold}/{n_splits} ==========")

    fold_val_mfcc   = X_mfcc_test_np[va_idx]
    fold_val_chroma = X_chroma_test_np[va_idx]
    fold_val_y      = y_test_np[va_idx]

    fold_train_mfcc_raw   = np.concatenate([X_mfcc_train_np,   X_mfcc_test_np[tr_idx]],   axis=0)
    fold_train_chroma_raw = np.concatenate([X_chroma_train_np, X_chroma_test_np[tr_idx]], axis=0)
    fold_train_y_raw      = np.concatenate([y_train_np,        y_test_np[tr_idx]],         axis=0)

    fold_train_mfcc, fold_train_chroma, fold_train_y = cap_per_class(
        fold_train_mfcc_raw, fold_train_chroma_raw, fold_train_y_raw, cap=TARGET_COUNT
    )

    print(f"  Train: {len(fold_train_y)} segs | Val: {len(fold_val_y)} segs (raw/imbalanced)")
    for cls in np.unique(fold_train_y):
        print(f"    class {cls}: {(fold_train_y==cls).sum()} train | {(fold_val_y==cls).sum()} val")

    fold_train_loader = DataLoader(
        DualFeatureDataset(
            torch.from_numpy(fold_train_mfcc).float(),
            torch.from_numpy(fold_train_chroma).float(),
            torch.from_numpy(fold_train_y).long(),
        ),
        batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True
    )
    fold_val_loader = DataLoader(
        DualFeatureDataset(
            torch.from_numpy(fold_val_mfcc).float(),
            torch.from_numpy(fold_val_chroma).float(),
            torch.from_numpy(fold_val_y).long(),
        ),
        batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True
    )

    # fuse_dim=128  [CHANGED from Lima's 64]
    model = DualChannelCNNLSTM_WithAttentionFusion(
        num_classes=num_classes,
        dropout_p=0.4,
        fuse_dim=128,
    ).to(device)

    fold_class_weights = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(num_classes),
        y=fold_train_y
    )
    fold_class_weights = torch.tensor(fold_class_weights, dtype=torch.float32).to(device)

    # Per-class focal gamma  [CHANGED from Lima's uniform gamma=1.5]
    criterion = PerClassFocalLoss(
        class_gammas=CLASS_GAMMAS.to(device),
        weight=fold_class_weights
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=6, min_lr=1e-6
    )
    early = EarlyStopping(patience=25, min_delta=1e-4)

    history = {
        "train_loss": [], "train_acc": [], "train_f1": [],
        "val_loss":   [], "val_acc":   [], "val_f1":   [],
        "lr":         [],
        "train_w_cnn": [], "train_w_lstm": [], "train_w_inter": [],
        "val_w_cnn":   [], "val_w_lstm":   [], "val_w_inter":   [],
    }

    best_this_fold       = float("inf")
    best_state_this_fold = None

    for epoch in range(1, epochs + 1):
        (train_loss, train_acc, train_bal_acc, train_prec, train_rec, train_f1,
         train_w_cnn, train_w_lstm, train_w_inter) = run_epoch(model, fold_train_loader, optimizer=optimizer)

        (val_loss, val_acc, val_bal_acc, val_prec, val_rec, val_f1,
         val_w_cnn, val_w_lstm, val_w_inter) = run_epoch(model, fold_val_loader, optimizer=None)

        scheduler.step(val_loss)
        lr = optimizer.param_groups[0]["lr"]

        history["lr"].append(lr)
        history["train_loss"].append(train_loss);      history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc);        history["val_acc"].append(val_acc)
        history["train_f1"].append(train_f1);          history["val_f1"].append(val_f1)
        history["train_w_cnn"].append(train_w_cnn);    history["val_w_cnn"].append(val_w_cnn)
        history["train_w_lstm"].append(train_w_lstm);  history["val_w_lstm"].append(val_w_lstm)
        history["train_w_inter"].append(train_w_inter); history["val_w_inter"].append(val_w_inter)

        print(
            f"Fold {fold} | Epoch {epoch:03d} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} "
            f"bal_acc {train_bal_acc:.4f} f1 {train_f1:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f} "
            f"bal_acc {val_bal_acc:.4f} f1 {val_f1:.4f} | "
            f"attn(cnn={val_w_cnn:.3f}, lstm={val_w_lstm:.3f}, inter={val_w_inter:.3f}) | "
            f"lr {lr:.2e}"
        )

        if val_loss <= best_this_fold - early.min_delta:
            best_this_fold       = val_loss
            best_state_this_fold = {k: v.detach().cpu().clone()
                                    for k, v in model.state_dict().items()}

        if early.step(val_loss):
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_state_this_fold)
    model.eval()

    fold_preds, fold_targets = [], []
    with torch.no_grad():
        for mfcc, chroma, y in fold_val_loader:
            logits = model(mfcc.to(device), chroma.to(device))
            fold_preds.append(torch.argmax(logits, dim=1).cpu().numpy())
            fold_targets.append(y.numpy())

    fold_preds   = np.concatenate(fold_preds)
    fold_targets = np.concatenate(fold_targets)
    all_cv_preds.extend(fold_preds)
    all_cv_targets.extend(fold_targets)

    fold_summaries.append({
        "fold":          fold,
        "best_val_loss": best_this_fold,
        "best_val_acc":  max(history["val_acc"]) if history["val_acc"] else float("nan"),
        "best_val_f1":   max(history["val_f1"])  if history["val_f1"]  else float("nan"),
        "history":       history,
    })

    if best_this_fold < best_fold_loss and best_state_this_fold is not None:
        best_fold_loss  = best_this_fold
        best_fold_state = best_state_this_fold


print("\n========== CV Summary ==========")
for s in fold_summaries:
    print(
        f"Fold {s['fold']}: best val loss={s['best_val_loss']:.4f} | "
        f"best val acc={s['best_val_acc']:.4f} | "
        f"best val f1={s['best_val_f1']:.4f}"
    )
mean_acc = np.nanmean([s["best_val_acc"] for s in fold_summaries])
mean_f1  = np.nanmean([s["best_val_f1"]  for s in fold_summaries])
print(f"Mean(best val acc) across folds: {mean_acc:.4f}")
print(f"Mean(best val f1 ) across folds: {mean_f1:.4f}")

all_cv_preds   = np.array(all_cv_preds)
all_cv_targets = np.array(all_cv_targets)
cm_cv      = confusion_matrix(all_cv_targets, all_cv_preds)
cm_cv_norm = np.divide(
    cm_cv.astype("float"),
    cm_cv.sum(axis=1, keepdims=True),
    where=cm_cv.sum(axis=1, keepdims=True) != 0,
)
class_names = ["Bronchiectasis", "Bronchiolitis", "COPD", "Healthy", "Pneumonia", "URTI"]
annot_cv = np.empty_like(cm_cv).astype(str)
for i in range(cm_cv.shape[0]):
    for j in range(cm_cv.shape[1]):
        count   = cm_cv[i, j]
        percent = cm_cv_norm[i, j] * 100
        annot_cv[i, j] = "0" if count == 0 else f"{count}\n({percent:.1f}%)"
plt.figure(figsize=(8, 6))
sns.heatmap(cm_cv, annot=annot_cv, fmt="", cmap="viridis",
            xticklabels=class_names, yticklabels=class_names, cbar=True)
plt.title("Confusion Matrix (Cross-Validation)")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.show()

if best_fold_state is not None:
    torch.save(best_fold_state, checkpoint_path)
    print("Saved best-fold checkpoint to:", checkpoint_path)
else:
    print("WARNING: best_fold_state is None — no checkpoint saved.")

## 12. Training Curves (Last Fold)

In [ ]:
last = fold_summaries[-1]["history"]

plt.figure(figsize=(10, 5))
plt.plot(last["train_loss"], label="Train loss")
plt.plot(last["val_loss"],   label="Val/Test loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.title("Loss Curves"); plt.show()

plt.figure(figsize=(10, 5))
plt.plot(last["train_acc"], label="Train acc")
plt.plot(last["val_acc"],   label="Val/Test acc")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.legend(); plt.title("Accuracy Curves"); plt.show()

plt.figure(figsize=(10, 5))
plt.plot(last["train_f1"], label="Train macro-F1")
plt.plot(last["val_f1"],   label="Val/Test macro-F1")
plt.xlabel("Epoch"); plt.ylabel("Macro-F1")
plt.legend(); plt.title("Macro-F1 Curves"); plt.show()

plt.figure(figsize=(10, 5))
plt.plot(last["val_w_cnn"],   label="Val attn w_cnn  (MFCC+Chroma CNN)")
plt.plot(last["val_w_lstm"],  label="Val attn w_lstm (MFCC+Chroma LSTM)")
plt.plot(last["val_w_inter"], label="Val attn w_inter (CNN+LSTM)")
plt.xlabel("Epoch"); plt.ylabel("Mean gate weight")
plt.legend(); plt.title("Attention/Gated Fusion Weight Trends (Validation)")
plt.show()

## 13. Unseen Test Set Evaluation

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for mfcc, chroma, y in unseen_loader:
        logits = model(mfcc.to(device), chroma.to(device))
        all_preds.append(torch.argmax(logits, dim=1).cpu().numpy())
        all_targets.append(y.numpy())

all_preds   = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

acc  = accuracy_score(all_targets, all_preds)
prec, rec, f1, _ = precision_recall_fscore_support(
    all_targets, all_preds, average="macro", zero_division=0
)
print(f"Unseen Accuracy: {acc:.4f}")
print(f"Unseen Macro Precision: {prec:.4f} | Recall: {rec:.4f} | F1: {f1:.4f}")
print("\nClassification report:")
print(classification_report(all_targets, all_preds, digits=4))

cm      = confusion_matrix(all_targets, all_preds)
cm_norm = cm.astype("float") / cm.sum(axis=1, keepdims=True)
annot = np.empty_like(cm).astype(str)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        count   = cm[i, j]
        percent = cm_norm[i, j] * 100
        annot[i, j] = "0" if count == 0 else f"{count}\n({percent:.1f}%)"

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=annot, fmt="", cmap="viridis",
            xticklabels=class_names, yticklabels=class_names, cbar=True)
plt.title("Confusion Matrix (Unseen)")
plt.xlabel("Predicted"); plt.ylabel("True")
plt.tight_layout(); plt.show()